# Bengal 2021 vs 2026 - two chartsTwo things the tables in the README can't show on their own:whether the turnout rise is real, and why so many seats flipped.Data comes from `bengal_elections.db`, built by `src/clean.py` and `src/load_db.py`.

In [ ]:
import sqlite3import numpy as npimport pandas as pdimport matplotlib.pyplot as pltconn = sqlite3.connect("../bengal_elections.db")# same look for both chartsplt.rcParams["figure.figsize"] = (8, 5)plt.rcParams["font.size"] = 11plt.rcParams["axes.spines.top"] = Falseplt.rcParams["axes.spines.right"] = FalseBLUE = "#2a78d6"ORANGE = "#eb6834"GREY = "#b8b6b0"

## 1. Did turnout actually go up?Turnout reads 93.6% in 2026 against 82.2% in 2021. But the electoral roll shrankat the same time, and turnout is votes divided by registered voters - so a smallerroll pushes the percentage up on its own.If that's what happened, seats where the roll shrank most should be the seats whereturnout "rose" most. One dot per seat, Falta excluded.

In [ ]:
q = '''SELECT e.ac_no, c.region,       MAX(CASE WHEN e.year=2021 THEN e.total_electors END) AS roll_2021,       MAX(CASE WHEN e.year=2026 THEN e.total_electors END) AS roll_2026,       MAX(CASE WHEN e.year=2021 THEN e.turnout_pct END) AS turnout_2021,       MAX(CASE WHEN e.year=2026 THEN e.turnout_pct END) AS turnout_2026FROM electorate eJOIN constituencies c ON c.ac_no = e.ac_noWHERE e.ac_no <> 144GROUP BY e.ac_no, c.region'''t = pd.read_sql(q, conn)t["roll_change"] = (t.roll_2026 - t.roll_2021) / t.roll_2021 * 100t["turnout_change"] = t.turnout_2026 - t.turnout_2021r = t.roll_change.corr(t.turnout_change)print("correlation:", round(r, 3))t[["roll_change", "turnout_change"]].describe().round(2)

In [ ]:
fig, ax = plt.subplots()ax.scatter(t.roll_change, t.turnout_change, s=28, color=BLUE, alpha=0.65,           edgecolors="white", linewidths=0.5)# trend lineslope, intercept = np.polyfit(t.roll_change, t.turnout_change, 1)xs = np.array([t.roll_change.min(), t.roll_change.max()])ax.plot(xs, slope * xs + intercept, color=ORANGE, linewidth=2)ax.axhline(0, color=GREY, linewidth=1)ax.axvline(0, color=GREY, linewidth=1)ax.grid(color="#e8e7e3", linewidth=0.8)ax.set_axisbelow(True)ax.set_xlabel("Change in electoral roll, 2021 to 2026 (%)")ax.set_ylabel("Change in turnout (percentage points)")ax.set_title("Where the electoral roll shrank, turnout 'rose'", fontsize=13, loc="left")ax.text(0.97, 0.95, f"r = {r:.2f}\n{len(t)} seats", transform=ax.transAxes,        ha="right", va="top", color="#52514e", fontsize=10)fig.tight_layout()fig.savefig("../figures/turnout_vs_roll.png", dpi=150)plt.show()

r = -0.84 across 293 seats. The seats that lost the most voters from the registerare the seats where turnout climbed most, which is what you'd expect if the rise ismostly a smaller denominator rather than more people voting.This doesn't explain *why* the roll shrank - that still needs checking againstwhatever revision the Election Commission ran.

## 2. Why did so many seats flip?TMC lost 7.4 points of vote share and 135 seats. That sounds disproportionate untilyou look at how close the 2021 seats were.A swing of 7.4 points moves a margin by roughly 15 - the winner loses what thechallenger gains. So every seat held by less than 15 points was in play.

In [ ]:
m21 = pd.read_sql("SELECT margin_pct_polled FROM winners WHERE year=2021", conn)THRESHOLD = 15n = (m21.margin_pct_polled < THRESHOLD).sum()print(n, "of 294 seats under", THRESHOLD, "points  =", round(n / 294 * 100), "%")m21.margin_pct_polled.describe().round(2)

In [ ]:
fig, ax = plt.subplots()bins = np.arange(0, 67.5, 2.5)   # 15 lands on a bin edgecounts, bins, patches = ax.hist(m21.margin_pct_polled, bins=bins,                                color=GREY, edgecolor="white", linewidth=0.8)# colour the bars below the thresholdfor patch, right_edge in zip(patches, bins[1:]):    if right_edge <= THRESHOLD:        patch.set_facecolor(ORANGE)ax.axvline(THRESHOLD, color="#52514e", linestyle="--", linewidth=1.5)ax.text(THRESHOLD + 1.5, ax.get_ylim()[1] * 0.88,        f"{n} of 294 seats had a margin\nunder 15 points - a 7.4pt swing\nputs all of them in play",        fontsize=10, color="#52514e")ax.grid(axis="y", color="#e8e7e3", linewidth=0.8)ax.set_axisbelow(True)ax.set_xlabel("2021 winning margin (% of votes polled)")ax.set_ylabel("Number of seats")ax.set_title("Two-thirds of 2021 seats sat inside the swing that followed",             fontsize=13, loc="left")fig.tight_layout()fig.savefig("../figures/margin_distribution_2021.png", dpi=150)plt.show()

197 of 294 seats were held by under 15 points. That's why a 7.4-point swing turnedinto a 135-seat loss - first-past-the-post doesn't care how big a margin is, onlywho's ahead, so a thin lead is worth the same as a landslide until it isn't.

In [ ]:
conn.close()